# <font color="#418FDE" size="6.5" uppercase>**Pipelines bauen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Wenden Skalierer, Imputer und Encoder passend auf numerische und kategoriale Spalten an. 
- Verbinden Spaltentransformationen mit ColumnTransformer und Pipeline. 
- Untersuchen Pipeline-Schritte, verschachtelte Parameter und Merkmalsnamen. 


## **1. Vorverarbeitung anwenden**

### **1.1. Skalierer gezielt einsetzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_01_01.jpg?v=1787681283" width="250">



>* Unterschiedliche Größenordnungen können Modelle verzerren
>* Skalierung macht numerische Merkmale vergleichbarer

>* Skalierung hilft abstands- und gradientenbasierten Modellen.
>* Skalierer passend zur Verteilung wählen.

>* Nur passende numerische Spalten skalieren
>* Skalierer auf Trainingsdaten lernen



In [ ]:
#@title Python-Code - Skalierer gezielt einsetzen

# Wir vergleichen Skalierer für numerische Immobilienmerkmale.
# Unterschiedliche Größenordnungen werden sichtbar und vergleichbar.
# Die Ausgabe zeigt passende Skalierung ohne Datenleck.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Kleine Beispieldaten zeigen typische numerische Größenordnungen.
housing_data = pd.DataFrame(
    {
        "area_m2": [45, 62, 80, 95, 120, 150, 210, 240],
        "year_built": [1955, 1970, 1985, 1998, 2005, 2012, 2018, 2022],
        "rooms": [2, 3, 3, 4, 4, 5, 6, 7],
        "price_eur": [180000, 240000, 310000, 390000, 520000, 690000, 980000, 1250000],
    }
)

# Die Zielspalte wird nicht als Eingabemerkmal skaliert.
feature_columns = ["area_m2", "year_built", "rooms"]
X = housing_data[feature_columns]
y = housing_data["price_eur"]

# Der Split trennt Trainingsdaten von späteren Testdaten.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

# Diese Prüfung macht die Beispielannahme ausdrücklich sichtbar.
if X_train.shape[0] < 2:
    raise ValueError("Für Skalierung werden mindestens zwei Trainingszeilen benötigt.")

# Jeder Skalierer lernt ausschließlich aus den Trainingsdaten.
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

# Wir betrachten eine Testwohnung vor und nach der Skalierung.
test_home = X_test.iloc[[0]]
scaled_rows = []

for scaler_name, scaler in scalers.items():
    scaler.fit(X_train)
    scaled_values = scaler.transform(test_home)[0]
    scaled_rows.append([scaler_name] + list(np.round(scaled_values, 2)))

# Eine kompakte Tabelle zeigt die Wirkung der Skalierer.
scaled_table = pd.DataFrame(
    scaled_rows,
    columns=["scaler", "area_m2", "year_built", "rooms"],
)

print(f"scikit-learn Version: {sklearn.__version__}")
print("Testwohnung original: " + str(test_home.iloc[0].to_dict()))
print(scaled_table.to_string(index=False))

# Die Grafik zeigt, wie Standardisierung Größenordnungen angleicht.
standard_scaler = StandardScaler()
standard_scaled_train = standard_scaler.fit_transform(X_train)

original_ranges = X_train.max() - X_train.min()
scaled_ranges = pd.Series(
    standard_scaled_train.max(axis=0) - standard_scaled_train.min(axis=0),
    index=feature_columns,
)

x_positions = np.arange(len(feature_columns))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x_positions - bar_width / 2, original_ranges, bar_width, label="Original")
ax.bar(x_positions + bar_width / 2, scaled_ranges, bar_width, label="Standardisiert")
ax.set_title("Merkmalsbereiche vor und nach StandardScaler")
ax.set_xlabel("Numerisches Merkmal")
ax.set_ylabel("Bereich im Trainingsdatensatz")
ax.set_xticks(x_positions)
ax.set_xticklabels(feature_columns)
ax.legend()
plt.show()



### **1.2. Fehlende Werte füllen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_01_02.jpg?v=1787681285" width="250">



>* Fehlende Werte gezielt per Imputation behandeln
>* Nur Trainingsdaten zum Füllen verwenden

>* Häufigste Kategorie kann Mehrheiten verstärken
>* Fehlend als eigene Kategorie kann Signal sein

>* Erst imputieren, dann skalieren oder kodieren
>* Imputation nur im Trainingsprozess lernen



In [ ]:
#@title Python-Code - Fehlende Werte füllen

# Dieses Beispiel füllt fehlende Werte gezielt.
# Numerische und kategoriale Spalten brauchen unterschiedliche Strategien.
# Die Ausgabe zeigt gelernte Ersatzwerte.

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import sklearn

# Ein kleiner Datensatz enthält absichtlich fehlende Werte.
data = pd.DataFrame(
    {
        "living_area_m2": [55.0, 80.0, None, 120.0, 75.0],
        "rooms": [2.0, None, 3.0, 5.0, 3.0],
        "heating": ["gas", "electric", None, "gas", "district"],
        "city": ["Berlin", None, "Hamburg", "Berlin", "Hamburg"],
    }
)

# Diese Spalten werden nach Datentyp getrennt verarbeitet.
numeric_features = ["living_area_m2", "rooms"]
categorical_features = ["heating", "city"]

# Numerische Lücken werden mit dem Median gefüllt.
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

# Kategoriale Lücken werden als eigene Kategorie markiert.
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="unbekannt")),
        ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
    ]
)

# ColumnTransformer wendet passende Schritte je Spaltengruppe an.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# Die Vorverarbeitung lernt Ersatzwerte nur aus diesen Daten.
transformed = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()

# Eine einfache Prüfung schützt vor unerwarteten Formen.
if transformed.shape[0] != len(data):
    raise ValueError("Die Zeilenzahl passt nach der Transformation nicht.")

# Gelernte Ersatzwerte werden aus den Pipeline-Schritten gelesen.
numeric_imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]
cat_imputer = preprocessor.named_transformers_["cat"].named_steps["imputer"]

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Originale Form: {data.shape[0]} Zeilen, {data.shape[1]} Spalten")
print(f"Transformierte Form: {transformed.shape[0]} Zeilen, {transformed.shape[1]} Spalten")
print(f"Median Wohnfläche: {numeric_imputer.statistics_[0]:.1f} m²")
print(f"Median Zimmer: {numeric_imputer.statistics_[1]:.1f}")
print(f"Kategorialer Ersatzwert: {cat_imputer.statistics_[0]}")
print("Erste Merkmale: " + ", ".join(feature_names[:5]))



### **1.3. Kategorien kodieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_01_03.jpg?v=1787681287" width="250">



>* Kategorien für Modelle numerisch darstellen
>* Nominale und ordinale Struktur beachten

>* One-Hot kodiert nominale Kategorien ohne Rangfolge
>* Viele Kategorien erhöhen Merkmale und Komplexität

>* Ordinale Kategorien nur begründet numerisch kodieren
>* Unbekannte und seltene Werte robust behandeln



In [ ]:
#@title Python-Code - Kategorien kodieren

# Dieses Beispiel kodiert kategoriale Merkmale für Modelle.
# One-Hot-Kodierung vermeidet künstliche Rangfolgen bei Kategorien.
# Die Ausgabe zeigt neue numerische Merkmale.

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Kleine Beispieldaten bleiben übersichtlich und vollständig im Speicher.
data = pd.DataFrame(
    {
        "city": ["Berlin", "Hamburg", "Berlin", "München"],
        "contract": ["monatlich", "jährlich", "jährlich", "monatlich"],
        "age": [28, 41, 35, 52],
    }
)

# Diese Prüfung macht die erwartete Tabellenform ausdrücklich sichtbar.
if data.shape != (4, 3):
    raise ValueError("Die Beispieltabelle sollte vier Zeilen und drei Spalten haben.")

# Nur nominale Textspalten werden per One-Hot-Encoding umgewandelt.
categorical_columns = ["city", "contract"]
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(sparse_output=False), categorical_columns)],
    remainder="passthrough",
)

# Der Transformer lernt die Kategorien und erzeugt numerische Spalten.
encoded_array = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()
encoded_data = pd.DataFrame(encoded_array, columns=feature_names)

# Die kompakte Ausgabe zeigt die Wirkung der Kodierung.
print("Originale kategoriale Spalten: city, contract")
print("Gelernte Merkmalsnamen:")
print(list(feature_names))
print("Kodierte Tabelle, erste vier Zeilen:")
print(encoded_data.round(0).head(4).to_string(index=False))



## **2. Spalten gezielt verbinden**

### **2.1. Seltene Kategorien bündeln**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_02_01.jpg?v=1787681289" width="250">



>* Kategorien sind oft ungleich häufig verteilt
>* Seltene Werte gemeinsam als „andere“ bündeln

>* Seltene Kategorien vor dem Kodieren bündeln
>* Weniger Merkmale, stabilere Modellmuster

>* Seltenheitsregeln werden reproduzierbar in Pipelines gelernt
>* Neue unbekannte Kategorien werden kontrolliert behandelt



In [ ]:
#@title Python-Code - Seltene Kategorien bündeln

# Dieses Beispiel bündelt seltene Kategorien automatisch.
# ColumnTransformer verbindet passende Spaltentransformationen sauber.
# Die Ausgabe zeigt kompakte neue Merkmalsnamen.

import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Kleine Beispieldaten enthalten häufige und seltene Städte.
data = pd.DataFrame(
    {
        "age": [22, 35, 41, 29, 52, 31, 46, 27, 38, 44, 33, 57],
        "city": ["Berlin", "Berlin", "Hamburg", "Berlin", "Kiel", "Ulm", "Hamburg", "Berlin", "Bonn", "Hamburg", "Kiel", "Passau"],
        "device": ["Smartphone", "Laptop", "Smartphone", "Tablet", "Smartphone", "Laptop", "Smartphone", "Smartphone", "Konsole", "Laptop", "Tablet", "Smartwatch"],
    }
)

# Diese Prüfung macht die Beispielannahme sichtbar.
if data.shape != (12, 3):
    raise ValueError("Die Beispieldaten sollten 12 Zeilen und 3 Spalten haben.")

# Numerische Spalten werden imputiert und skaliert.
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Seltene Kategorien werden vor dem One-Hot-Encoding gebündelt.
categorical_pipeline = Pipeline(
    steps=[("encoder", OneHotEncoder(min_frequency=3, handle_unknown="infrequent_if_exist", sparse_output=False))]
)

# ColumnTransformer wendet beide Pipelines spaltengenau an.
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipeline, ["age"]), ("cat", categorical_pipeline, ["city", "device"])]
)

# Fit lernt Häufigkeiten nur aus diesen Trainingsdaten.
transformed = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()

# Die Merkmalsnamen zeigen die gebündelten Kategorien.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Ursprüngliche Spalten: {data.shape[1]}")
print(f"Merkmale nach Transformation: {transformed.shape[1]}")
print("Neue Merkmale: " + ", ".join(feature_names))

# Die Grafik vergleicht Rohkategorien mit erzeugten Merkmalen.
counts = pd.Series(
    {"Rohkategorien": data["city"].nunique() + data["device"].nunique(), "Kodierte Merkmale": transformed.shape[1] - 1}
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color=["#7aa6c2", "#f2a65a"])
ax.set_title("Seltene Kategorien bündeln")
ax.set_xlabel("Darstellung")
ax.set_ylabel("Anzahl")
plt.show()



### **2.2. Spalten gezielt transformieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_02_02.jpg?v=1787681291" width="250">



>* Verschiedene Spaltentypen brauchen passende Vorbereitung
>* Numerische Werte skalieren, Kategorien codieren

>* ColumnTransformer verarbeitet Spaltengruppen parallel
>* Klare Zuordnung reduziert Fehler

>* Pipelines sichern konsistente Verarbeitung ohne Datenleckage
>* Transformationen bleiben nachvollziehbar und reproduzierbar



In [ ]:
#@title Python-Code - Spalten gezielt transformieren

# Dieses Beispiel transformiert ausgewählte Tabellenspalten gezielt.
# Numerische und kategoriale Spalten erhalten passende Schritte.
# Die Ausgabe zeigt neue Merkmale nach ColumnTransformer.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Eine kleine Tabelle macht die Spaltentypen sichtbar.
housing = pd.DataFrame(
    {
        "area_m2": [45, 80, 120, None, 65],
        "rooms": [2, 3, 5, 4, None],
        "district": ["Nord", "Sued", "Nord", "West", None],
        "building_type": ["Altbau", "Neubau", "Neubau", "Altbau", "Altbau"],
    }
)

# Diese Prüfung verhindert unklare Spaltennamen im Beispiel.
expected_columns = ["area_m2", "rooms", "district", "building_type"]
if list(housing.columns) != expected_columns:
    raise ValueError("Die Beispieltabelle hat unerwartete Spalten.")

# Numerische Spalten werden ersetzt und anschließend skaliert.
numeric_features = ["area_m2", "rooms"]
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer()), ("scaler", StandardScaler())]
)

# Kategoriale Spalten werden ersetzt und anschließend codiert.
categorical_features = ["district", "building_type"]
categorical_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent")),
           ("encoder", OneHotEncoder(sparse_output=False))]
)

# Der ColumnTransformer verbindet beide Spaltenwege parallel.
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipeline, numeric_features),
                  ("cat", categorical_pipeline, categorical_features)]
)

# Fit lernt nur die Vorverarbeitungsregeln aus der Tabelle.
transformed_array = preprocessor.fit_transform(housing)
feature_names = preprocessor.get_feature_names_out()

# Eine kleine Ergebnisansicht zeigt die gemeinsame Merkmalsmatrix.
transformed_table = pd.DataFrame(transformed_array, columns=feature_names)
preview = transformed_table.iloc[:3, :5].round(2)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Originale Form: {housing.shape}")
print(f"Transformierte Form: {transformed_table.shape}")
print("Erste transformierte Merkmale:")
print(preview.to_string(index=False))



### **2.3. Pipeline Schritte verbinden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_02_03.jpg?v=1787681293" width="250">



>* Pipeline verbindet Vorbereitung und Modelltraining
>* Gleiche Verarbeitung für Training und neue Daten

>* ColumnTransformer bereitet Spaltengruppen passend vor
>* Pipeline schützt vor Datenleckage bei Validierung

>* Pipelines machen Abläufe wartbar und wiederverwendbar
>* Gleiche Regeln für Training und Einsatz



In [ ]:
#@title Python-Code - Pipeline Schritte verbinden

# Diese Pipeline verbindet Vorverarbeitung und Modell.
# ColumnTransformer behandelt Spaltengruppen gezielt unterschiedlich.
# Die Ausgabe zeigt Schritte und Merkmalsnamen.

import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Kleine Rohdaten enthalten numerische und kategoriale Spalten.
data = pd.DataFrame(
    {
        "area_m2": [55, 80, 120, 45, 95, 70, 110, 60],
        "rooms": [2, 3, 4, 2, None, 3, 5, None],
        "district": ["Zentrum", "Nord", "Sued", "Nord", "Zentrum", "Sued", "Nord", None],
        "building_type": ["Altbau", "Neubau", "Neubau", "Altbau", "Altbau", "Neubau", "Altbau", "Neubau"],
    }
)

target = pd.Series([0, 1, 1, 0, 1, 0, 1, 0], name="high_price")

# Die Aufteilung verhindert Datenleckage beim Anpassen der Vorverarbeitung.
X_train, X_test, y_train, y_test = train_test_split(
    data,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)

# Numerische Spalten werden imputiert und anschließend skaliert.
numeric_features = ["area_m2", "rooms"]
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Kategoriale Spalten werden imputiert und anschließend kodiert.
categorical_features = ["district", "building_type"]
categorical_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

# ColumnTransformer verbindet beide Spaltenwege zu einem Vorverarbeitungsschritt.
preprocessor = ColumnTransformer(
    [("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)]
)

# Die Gesamtpipeline setzt Vorverarbeitung direkt vor das Modell.
model_pipeline = Pipeline(
    [("preprocess", preprocessor), ("model", LogisticRegression(random_state=42))]
)

# Die gesamte Pipeline wird mit Rohdaten trainiert.
model_pipeline.fit(X_train, y_train)

# Eine einfache Prüfung macht die erwartete Struktur sichtbar.
feature_names = model_pipeline.named_steps["preprocess"].get_feature_names_out()
if len(feature_names) == 0:
    raise ValueError("Die Pipeline hat keine Merkmale erzeugt.")

# Kurze Ausgaben zeigen Schritte, Parameter und Ergebnis.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Pipeline-Schritte: {list(model_pipeline.named_steps.keys())}")
print(f"Vorverarbeitete Merkmale: {len(feature_names)}")
print(f"Beispiel-Merkmale: {list(feature_names[:5])}")
print(f"Testgenauigkeit: {model_pipeline.score(X_test, y_test):.2f}")

# Die Grafik zeigt, wie viele Merkmale je Spaltengruppe entstehen.
feature_counts = pd.Series(
    {"numerisch": len(numeric_features), "nach Kodierung": len(feature_names)}
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(feature_counts.index, feature_counts.values, color=["steelblue", "orange"])
ax.set_title("Merkmale vor und nach dem ColumnTransformer")
ax.set_xlabel("Darstellung")
ax.set_ylabel("Anzahl Merkmale")
plt.show()



## **3. Pipeline prüfen**

### **3.1. Schritte gezielt inspizieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_03_01.jpg?v=1787681298" width="250">



>* Pipeline-Schritte einzeln nachvollziehen
>* Fehlerquellen systematisch früher erkennen

>* Zwischenergebnisse und gelernte Parameter prüfen
>* Unerwartete Kategorien zeigen Datenprobleme

>* Klare Schrittnamen erleichtern Teamarbeit und Dokumentation
>* Inspektion macht Pipelines robuster und erklärbarer



In [ ]:
#@title Python-Code - Schritte gezielt inspizieren

# Diese Übung macht Pipeline-Schritte sichtbar.
# Wir prüfen Transformer, Parameter und Merkmalsnamen.
# Die Ausgabe zeigt gelernte Zwischenergebnisse.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Kleine Beispieldaten enthalten numerische und kategoriale Spalten.
data = pd.DataFrame(
    {
        "age": [22, 38, None, 45, 29, 52, 41, 35],
        "income": [32000, 58000, 41000, None, 39000, 76000, 61000, 48000],
        "city": ["Berlin", "Köln", "Berlin", "München", None, "Köln", "Berlin", "München"],
        "segment": ["neu", "treu", "neu", "treu", "neu", "treu", "treu", "neu"],
    }
)

# Das Ziel ist hier nur für ein vollständiges Modell nötig.
target = [0, 1, 0, 1, 0, 1, 1, 0]

# Diese Listen steuern die parallelen Verarbeitungspfade.
numeric_features = ["age", "income"]
categorical_features = ["city", "segment"]

# Numerische Werte werden ersetzt und anschließend skaliert.
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Kategoriale Werte werden ersetzt und danach One-Hot-kodiert.
categorical_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder())]
)

# Der ColumnTransformer verbindet beide Verarbeitungspfade.
preprocessor = ColumnTransformer(
    [("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)]
)

# Die gesamte Pipeline endet mit einem einfachen Klassifikationsmodell.
model = Pipeline(
    [("preprocess", preprocessor), ("classifier", LogisticRegression(random_state=42))]
)

# Nach fit sind die inneren Schritte inspizierbar.
model.fit(data, target)

# Eine einfache Prüfung schützt vor unerwarteten Spaltenzahlen.
feature_names = model.named_steps["preprocess"].get_feature_names_out()
if len(feature_names) != 7:
    raise ValueError("Die erwartete Anzahl erzeugter Merkmale stimmt nicht.")

# Verschachtelte Schritte erreicht man über ihre Namen.
fitted_preprocessor = model.named_steps["preprocess"]
fitted_numeric = fitted_preprocessor.named_transformers_["num"]
fitted_categorical = fitted_preprocessor.named_transformers_["cat"]

# Gelernte Werte zeigen, was die Transformer gespeichert haben.
median_values = fitted_numeric.named_steps["imputer"].statistics_
scaler_means = fitted_numeric.named_steps["scaler"].mean_
encoder_categories = fitted_categorical.named_steps["encoder"].categories_

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Pipeline-Schritte: {list(model.named_steps.keys())}")
print(f"Numerische Mediane: {[round(value, 1) for value in median_values]}")
print(f"Scaler-Mittelwerte: {[round(value, 1) for value in scaler_means]}")
print(f"Kategorien für city: {list(encoder_categories[0])}")
print(f"Erste Merkmalsnamen: {list(feature_names[:5])}")
print(f"Parameterzugriff: {model.get_params()['preprocess__num__imputer__strategy']}")



### **3.2. Merkmalsnamen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_03_02.jpg?v=1787681295" width="250">



>* Merkmalsnamen zeigen Pipeline-Veränderungen sichtbar.
>* Sie verbinden Datenbedeutung mit Modellrepräsentation.

>* ColumnTransformer erzeugt strukturierte neue Merkmalsnamen
>* Namen machen Modellanalysen fachlich verständlich

>* Merkmalsnamen zeigen Fehler in der Vorverarbeitung
>* Abgleich verbessert Fehlersuche und Modellinterpretation



In [ ]:
#@title Python-Code - Merkmalsnamen verstehen

# Dieses Beispiel macht erzeugte Merkmalsnamen sichtbar.
# ColumnTransformer benennt numerische und kategoriale Ausgaben.
# Die Ausgabe zeigt Herkunft und neue Spalten.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Kleine Rohdaten zeigen numerische und kategoriale Spalten.
data = pd.DataFrame(
    {
        "age": [25, 40, None, 35],
        "income": [32000, 58000, 45000, None],
        "city": ["Berlin", "Hamburg", "Berlin", None],
        "contract": ["monthly", "yearly", "monthly", "yearly"],
    }
)

# Diese Listen steuern die Zweige im ColumnTransformer.
numeric_features = ["age", "income"]
categorical_features = ["city", "contract"]

# Numerische Werte werden ersetzt und anschließend skaliert.
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer()), ("scaler", StandardScaler())]
)

# Kategorien werden ersetzt und in Indikatorspalten zerlegt.
categorical_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent")),
           ("encoder", OneHotEncoder(sparse_output=False))]
)

# Der ColumnTransformer verbindet beide Verarbeitungszweige.
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipeline, numeric_features),
                  ("cat", categorical_pipeline, categorical_features)]
)

# Fit lernt Imputerwerte, Skalierung und vorhandene Kategorien.
transformed_data = preprocessor.fit_transform(data)

# Eine einfache Prüfung schützt vor unerwarteten Spaltenzahlen.
feature_names = preprocessor.get_feature_names_out()
if transformed_data.shape[1] != len(feature_names):
    raise ValueError("Die Anzahl der Merkmalsnamen passt nicht.")

# Die Namen zeigen Transformationszweig und ursprüngliche Bedeutung.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Rohdaten-Spalten: {list(data.columns)}")
print(f"Transformierte Matrix: {transformed_data.shape[0]} Zeilen, {transformed_data.shape[1]} Merkmale")
print("Erzeugte Merkmalsnamen:")
for name in feature_names:
    print(f"- {name}")

# Ein kleiner Ausschnitt verbindet Namen mit transformierten Werten.
preview = pd.DataFrame(transformed_data, columns=feature_names).round(2)
print("Erste zwei transformierte Zeilen:")
print(preview.head(2).iloc[:, :5])



### **3.3. End to End Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_03_03.jpg?v=1787681296" width="250">



>* Pipeline als gesamten Arbeitsprozess verstehen
>* Transformationen prüfen und Ergebnisse fachlich erklären

>* Pipeline-Schritte gezielt prüfen
>* Gelernte Werte und Probleme erkennen

>* Merkmalsnamen mit Rohdaten verständlich verknüpfen
>* Pipelines sichern reproduzierbare, erklärbare Analysen



In [ ]:
#@title Python-Code - End to End Projekt

# Wir prüfen eine vollständige scikit-learn Pipeline.
# Verschachtelte Schritte zeigen gelernte Vorverarbeitung.
# Merkmalsnamen verbinden Rohdaten mit Modellspalten.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Kleine Rohdaten enthalten Zahlen, Kategorien und fehlende Werte.
data = pd.DataFrame(
    {
        "income_eur": [32000, 54000, np.nan, 41000, 69000, 28000, 76000, 50000],
        "age_years": [25, 44, 31, np.nan, 52, 23, 58, 39],
        "contract": ["befristet", "unbefristet", "befristet", "frei", "unbefristet", "frei", "unbefristet", None],
        "city": ["Berlin", "Hamburg", "Berlin", "Köln", "Hamburg", "Köln", "Berlin", "Hamburg"],
    }
)

target = np.array([0, 1, 0, 0, 1, 0, 1, 1])

# Die Prüfung schützt vor versehentlich falschen Zeilenzahlen.
if len(data) != len(target):
    raise ValueError("Daten und Zielwerte müssen gleich viele Zeilen haben.")

numeric_features = ["income_eur", "age_years"]
categorical_features = ["contract", "city"]

# Numerische Spalten werden imputiert und anschließend skaliert.
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Kategoriale Spalten werden imputiert und danach One-Hot-kodiert.
categorical_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
)

preprocessor = ColumnTransformer(
    [("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)]
)

model_pipeline = Pipeline(
    [("preprocess", preprocessor), ("model", LogisticRegression(random_state=42))]
)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, test_size=0.25, random_state=42, stratify=target
)

# Fit lernt Imputerwerte, Skalierungswerte, Kategorien und Modellgewichte.
model_pipeline.fit(X_train, y_train)

predictions = model_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

fitted_preprocessor = model_pipeline.named_steps["preprocess"]
fitted_numeric = fitted_preprocessor.named_transformers_["num"]
fitted_categorical = fitted_preprocessor.named_transformers_["cat"]

numeric_imputer = fitted_numeric.named_steps["imputer"]
numeric_scaler = fitted_numeric.named_steps["scaler"]
categorical_encoder = fitted_categorical.named_steps["encoder"]

feature_names = fitted_preprocessor.get_feature_names_out()
model_coefficients = model_pipeline.named_steps["model"].coef_[0]

# Die wichtigsten Modellspalten werden nach Koeffizientbetrag sortiert.
importance = pd.DataFrame(
    {"feature": feature_names, "coefficient": model_coefficients}
)

importance["abs_coefficient"] = importance["coefficient"].abs()
importance = importance.sort_values("abs_coefficient", ascending=False)
top_features = importance.head(5).copy()

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit: {accuracy:.2f}")
print(f"Pipeline-Schritte: {list(model_pipeline.named_steps.keys())}")
print(f"Gelernte numerische Ersatzwerte: {np.round(numeric_imputer.statistics_, 1)}")
print(f"Gelernte Skalierer-Mittelwerte: {np.round(numeric_scaler.mean_, 1)}")
print(f"Erzeugte Merkmale: {len(feature_names)}")
print(f"Beispiel für verschachtelten Parameter: {model_pipeline.get_params()['preprocess__num__imputer__strategy']}")

# Das Diagramm zeigt, welche erzeugten Merkmale stark wirken.
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top_features["feature"], top_features["coefficient"])
ax.set_title("Wichtigste erzeugte Pipeline-Merkmale")
ax.set_xlabel("Logistische Regressionskoeffizienten")
ax.set_ylabel("Merkmalsname nach Vorverarbeitung")
ax.invert_yaxis()
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Pipelines bauen**</font>


In this lecture, you learned to:
- Wenden Skalierer, Imputer und Encoder passend auf numerische und kategoriale Spalten an. 
- Verbinden Spaltentransformationen mit ColumnTransformer und Pipeline. 
- Untersuchen Pipeline-Schritte, verschachtelte Parameter und Merkmalsnamen. 

In the next Module (Module 10), we will go over 'Klassische Modelle'